## Week 7
**Name: Ashley Hernandez Date: 10/12**

In [17]:
import requests #brings in functions needed to grab the data from the web and deal with responses 

#variables that make up the components of the URL to query MET API
base_url = "https://collectionapi.metmuseum.org" #same no matter what
search_path = "/public/collection/v1.1/search" #this is the "endpoint" that deals with searching object IDS in the question 
search_query = "?q=John Singer Sargent" #parametters of the API query, q means: search term, 
artist_or_culture = "&artistOrCulture=True" #empty string will not be appended
limit = "&limit=500"
offset = "&offset=0"

url = f'{base_url}{search_path}{search_query}{artist_or_culture}{limit}{offset}'
search_response = requests.get(url)

print(url)  

https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=John Singer Sargent&artistOrCulture=True&limit=500&offset=0


In [18]:
#searching for a specific artist
print(search_response.ok)

True


In [19]:
met_objects = search_response.json()
print(met_objects.keys())
#print(met_objects["objectIDs"])
complete_jss_objects = met_objects["objectIDs"]

print(len(complete_jss_objects))
#print(len(met_objects["objectIDs"]))
print("out of")
print(met_objects["total"])

dict_keys(['total', 'objectIDs'])
500
out of
1485


In [20]:
limit = "&limit=500"
offset = "&offset=500"

url = f'{base_url}{search_path}{search_query}{artist_or_culture}{limit}{offset}'
search_response = requests.get(url)

print(search_response.ok)
complete_jss_objects.extend(search_response.json()["objectIDs"])

print(len(complete_jss_objects))

True
1000


In [24]:
from time import sleep 

object_path = "/public/collection/v1/objects/"

all_jss_details=[]

for jss_artwork in complete_jss_objects[:10]:
    url = f"{base_url}{object_path}{jss_artwork}"
    response_object = requests.get(url)
    all_jss_details.append(response_object.json())
    print(f"grabbed details from object id{jss_artwork}")
    sleep(0.5)

print(all_jss_details[0]['artistDisplayName'])

grabbed details from object id12234
grabbed details from object id12462
grabbed details from object id12317
grabbed details from object id14385
grabbed details from object id12365
grabbed details from object id15451
grabbed details from object id12139
grabbed details from object id12107
grabbed details from object id12304
grabbed details from object id12440
John Singer Sargent


## NYTimes API and Web scraping Part 1


In [26]:
import requests # request data from websites
from time import sleep # implement time delays in the code
from getpass import getpass # password protection feature

key = getpass("Paste your API key here!")

Paste your API key here! ········


In [28]:
query = "tv"      #search term in the metadata

begin_date = "20260901" #begin date for our search in September 
end_date = "20260930"  #end date for our search in September 

url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&api-key={key}"
response = requests.get(url)

print(response.ok)

True


Pulling the actual documents out of the API response:
- first we need to parse the response as JSON, then we need to find the key that contains the article info

In [31]:
parsed_nytimes_response = response.json() #intrepret response as Python-readable JSON
print(parsed_nytimes_response.keys()) #tells me what keys are in the raw response
print(parsed_nytimes_response['response'].keys())  #tells me what keys are in the "response" parent
nytimes_docs= parsed_nytimes_response['response']['docs'] #tells me what keys are in the "docs" keysm, which is a child of "response"
print(len(nytimes_docs))

dict_keys(['status', 'copyright', 'response'])
dict_keys(['docs', 'metadata'])
10


In [37]:
print(nytimes_docs[1]['headline'])
print(nytimes_docs[1]['headline']['main'])
print(nytimes_docs[1]['byline'])

{'main': '‘Ghost Adventures,’ and 7 More Shows to Watch on TV This Week', 'kicker': '', 'print_headline': 'This Week on TV'}
‘Ghost Adventures,’ and 7 More Shows to Watch on TV This Week
{'original': 'By Geordon Wollner'}


In [38]:
print(parsed_nytimes_response['response']['metadata'])

{'hits': 567, 'offset': 0, 'time': 179}


In [ ]:
results = []

for page in range(0, 5):
    url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&page={page}&api-key={key}"
    response = requests.get(url)

    if response.ok:
        parsed = response.json()
        articles = parsed["response"]["docs"]
        results.append(articles)
    else:
        print("page", page, "failed with status code", response.status_code)

    sleep(12)

## Web scraping


In [39]:
import csv
import json
import os
import requests
from time import sleep
from bs4 import BeautifulSoup

Below are some example HTML that we can traverse using the Beautiful Soup functions!

In [42]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python


In [43]:
title = soup.find("h1") #find the first element that uses the h1 tag
print(title)
print(title.get_text())

<h1>Adeline Ravoux</h1>


In [44]:
paragraph_elements = soup.find_all("p")
print(paragraph_elements)

[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]


In [48]:
artist = soup.find("p", class_= "artist")
print(artist)

<p class="artist">Vincent van Gogh</p>


In [49]:
print(artist.get_text())

Vincent van Gogh


In [50]:
date = soup.find("p", class_="date")
print(date.get_text())

1890


In [ ]:
date = artist.find_next_sibling() # finds the next element that is a "sibling" of the referenced element 

## Understanding robots permissions

We can view the permissions that a website grants to do things like automatically access data with a "bot" by looking at them.
'robots.txt' file

(clevelandart.org/robots.txt)

We can see that the cleveland art museum robots policy allows for any user agent to access most of the website, with a 10 second crawl delay


In [52]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text